In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/models/nnytemirlan09a/nllb-cgh/transformers/default/1/chagatai_NIRM/nllb200_model/nllb200_M_v7_DATA1500/final_v1'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/nnytemirlan09a/nllb-cgh/transformers/default/1/chagatai_NIRM/nllb200_model/nllb200_M_v7_DATA1500/final_v1/config.json
/kaggle/input/models/nnytemirlan09a/nllb-cgh/transformers/default/1/chagatai_NIRM/nllb200_model/nllb200_M_v7_DATA1500/final_v1/training_args.bin
/kaggle/input/models/nnytemirlan09a/nllb-cgh/transformers/default/1/chagatai_NIRM/nllb200_model/nllb200_M_v7_DATA1500/final_v1/tokenizer.json
/kaggle/input/models/nnytemirlan09a/nllb-cgh/transformers/default/1/chagatai_NIRM/nllb200_model/nllb200_M_v7_DATA1500/final_v1/tokenizer_config.json
/kaggle/input/models/nnytemirlan09a/nllb-cgh/transformers/default/1/chagatai_NIRM/nllb200_model/nllb200_M_v7_DATA1500/final_v1/model.safetensors
/kaggle/input/models/nnytemirlan09a/nllb-cgh/transformers/default/1/chagatai_NIRM/nllb200_model/nllb200_M_v7_DATA1500/final_v1/generation_config.json


In [6]:
"""
Chagatai Keyboard — Next Token Prediction с NLLB-200
=====================================================
Запускать на Kaggle с GPU (T4).
Модель: /kaggle/input/models/nnytemirlan09a/nllb-cgh/...

Два режима:
  A) word_completions(prefix)   — топ-K завершений слова (для клавиатуры)
  B) next_token_probs(sequence) — топ-K следующих токенов (языковая модель)
"""

# ── 0. Импорты ──────────────────────────────────────────────────────────────
import torch
from transformers import NllbTokenizer, M2M100ForConditionalGeneration

# ── 1. Константы ─────────────────────────────────────────────────────────────
MODEL_DIR = (
    "/kaggle/input/models/nnytemirlan09a/nllb-cgh/transformers/default/1/"
    "chagatai_NIRM/nllb200_model/nllb200_M_v7_DATA1500/final_v1"
)

# Chagatai — язык-источник и язык-цель (для seq2seq нам нужны оба)
SRC_LANG = "chg_Arab"
TGT_LANG = "chg_Arab"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


# ── 2. Загрузка модели ────────────────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = NllbTokenizer.from_pretrained(MODEL_DIR)

print("Loading model...")
model = M2M100ForConditionalGeneration.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)
model.eval()
if DEVICE == "cpu":
    model = model.to(DEVICE)

print("Model loaded ✓")


# ── 3. Подход A — завершение слова (для клавиатуры) ──────────────────────────
def word_completions(
    prefix: str,
    context: str = "",
    top_k: int = 5,
    max_new_tokens: int = 10,
    num_beams: int = 5,
) -> list[str]:
    full_input = (context + " " + prefix).strip() if context else prefix
    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(
        full_input,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(DEVICE)
    forced_bos = tokenizer.fairseq_tokens_to_ids.get(TGT_LANG) or tokenizer.convert_tokens_to_ids(TGT_LANG)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos,
            num_beams=num_beams,
            num_return_sequences=top_k,
            max_new_tokens=max_new_tokens,
            early_stopping=True,
            no_repeat_ngram_size=2,
            min_new_tokens=3,
        )
    results = []
    seen = set()
    
    for ids in output_ids:
        decoded = tokenizer.decode(ids, skip_special_tokens=True).strip()
    
        generated_part = decoded[len(full_input):].strip()
        first_word = generated_part.split()[0] if generated_part else ""
    
        if first_word.lower().startswith(prefix.lower()) and first_word not in seen:
            seen.add(first_word)
            results.append(first_word)
    
    return results[:top_k]


# ── 4. Подход B — вероятности следующего токена ───────────────────────────────

# ── 5. Demo ───────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "="*60)
    print("ПОДХОД A — word completions (keyboard style)")
    print("="*60)
    all_test = ['']

    test_prefixes = ["men", "kel",'Kiçik', "Men seni", 'کچيک بلام']
    for prefix in test_prefixes:
        completions = word_completions(prefix, top_k=5)
        print(f"  prefix='{prefix}' → {completions}")

    print("\n" + "="*60)
    print("ПОДХОД B — next token probabilities")
    print("="*60)

    test_contexts = ["Assalawma","Bügün", 'Māwar', 'bug', 'bu', 'sal', 'کچيک بلام']
    for ctx in test_contexts:
        top_tokens = next_token_probs(ctx, top_k=5)
        print(f"\n  context='{ctx}':")
        for token, prob in top_tokens:
            bar = "█" * int(prob * 30)
            print(f"    {token:<20} {prob:.4f}  {bar}")


# ── 6. Простой интерактивный цикл (keyboard simulator) ───────────────────────
def keyboard_loop():
    """
    Простой REPL для тестирования предиктивного ввода.
    Введи начало слова — получи предложения.
    """
    print("\n🎹 Chagatai Keyboard Simulator")
    print("  Введи prefix (или 'q' для выхода):\n")

    context = ""
    while True:
        prefix = input("  > ").strip()
        if prefix.lower() == "q":
            break
        if prefix == "":
            continue

        completions = word_completions(prefix, context=context, top_k=5)
        print(f"  Suggestions: {' | '.join(completions) if completions else '(нет)'}\n")

        # Опционально: принять одно из предложений как context
        choice = input("  Выбери (1-5) или Enter чтобы пропустить: ").strip()
        if choice.isdigit():
            idx = int(choice) - 1
            if 0 <= idx < len(completions):
                context += " " + completions[idx]
                print(f"  Context: '{context.strip()}'\n")


# Раскомментируй для интерактивного режима:
# keyboard_loop()

Device: cuda
Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Model loaded ✓

ПОДХОД A — word completions (keyboard style)
  prefix='men' → []
  prefix='kel' → []
  prefix='Kiçik' → []
  prefix='Men seni' → []
  prefix='کچيک بلام' → []

ПОДХОД B — next token probabilities

  context='Assalawma':
    ▁as                  0.4568  █████████████
    ▁As                  0.3342  ██████████
    ▁asos                0.1393  ████
    ▁av                  0.0130  
    ▁Es                  0.0064  

  context='Bügün':
    ▁bugun               0.6353  ███████████████████
    ▁bugungi             0.1764  █████
    ▁bug                 0.0887  ██
    ▁bu                  0.0406  █
    ▁Bugun               0.0214  

  context='Māwar':
    ▁Mav                 0.6050  ██████████████████
    ▁Ma                  0.1816  █████
    ▁Mo                  0.0793  ██
    ▁mav                 0.0599  █
    ▁ma                  0.0205  

  context='bug':
    ▁bug                 0.9761  █████████████████████████████
    ▁ox                  0.0047  
    ▁g             

In [ ]:
keyboard_loop()

In [ ]:
print(type(tokenizer))
print(dir(tokenizer))

In [ ]:
# Попробуй найти lang token напрямую через vocab
vocab = tokenizer.get_vocab()
chg_tokens = {k: v for k, v in vocab.items() if "chg" in k.lower()}
print(chg_tokens)

In [ ]:
pd.read_csv('/kaggle/input/models/nnytemirlan09a/nllb-cgh/transformers/default/1/dataset_chg/train_set.csv').to_csv('t.csv')

In [ ]:
pd.read_csv('t.csv')

In [ ]:
"""
Chagatai Keyboard — N-gram + Prefix Lookup
==========================================
Использует chg_Latn колонку из датасета.
Комбинирует:
  1. N-gram модель (bigram + trigram) — предсказание следующего слова по контексту
  2. Prefix lookup — фильтрация по введённым буквам
  3. Frequency fallback — если контекста нет, просто частотный список
"""

import re
import pandas as pd
from collections import defaultdict, Counter
from pathlib import Path


# ── 1. Загрузка и токенизация ──────────────────────────────────────────────

CSV_PATH = "/kaggle/working/t.csv"   # ← замени на свой путь к t.csv

def load_corpus(csv_path: str) -> list[list[str]]:
    df = pd.read_csv(csv_path)
    
    # Берём все тексты: и предложения и слова (слова дают словарь для prefix)
    all_texts = df["chg_Latn"].dropna().tolist()
    
    sentences = []
    for text in all_texts:
        tokens = tokenize(text)
        if tokens:
            sentences.append(tokens)
    
    return sentences


def tokenize(text: str) -> list[str]:
    """Токенизация по пробелам с лёгкой нормализацией."""
    text = text.strip()
    # Разбиваем по пробелам, сохраняем дефисные слова целиком
    tokens = text.split()
    # Убираем пустые и слишком короткие
    return [t for t in tokens if len(t) >= 1]


# ── 2. N-gram модель ──────────────────────────────────────────────────────

class NgramModel:
    def __init__(self, sentences: list[list[str]], n: int = 3):
        self.n = n
        self.unigram: Counter = Counter()
        self.bigram:  defaultdict[str, Counter] = defaultdict(Counter)
        self.trigram: defaultdict[tuple, Counter] = defaultdict(Counter)
        
        self._build(sentences)
    
    def _build(self, sentences):
        for tokens in sentences:
            # Unigram
            for t in tokens:
                self.unigram[t] += 1
            
            # Bigram
            for i in range(len(tokens) - 1):
                self.bigram[tokens[i]][tokens[i+1]] += 1
            
            # Trigram
            for i in range(len(tokens) - 2):
                ctx = (tokens[i], tokens[i+1])
                self.trigram[ctx][tokens[i+2]] += 1
        
        print(f"Corpus: {len(sentences)} sentences")
        print(f"Unigrams: {len(self.unigram)}, "
              f"Bigram contexts: {len(self.bigram)}, "
              f"Trigram contexts: {len(self.trigram)}")
    
    def predict(
        self,
        context: list[str],
        prefix: str = "",
        top_k: int = 5,
    ) -> list[tuple[str, float]]:
        """
        Предсказывает следующее слово.
        
        Args:
            context: Список предыдущих слов. Пример: ["Men", "seni"]
            prefix:  Введённые буквы текущего слова. Пример: "sev"
            top_k:   Сколько вернуть кандидатов.
        
        Returns:
            Список (слово, score) отсортированный по убыванию.
        """
        candidates: Counter = Counter()
        
        # Trigram (вес 3)
        if len(context) >= 2:
            ctx3 = (context[-2], context[-1])
            if ctx3 in self.trigram:
                for word, cnt in self.trigram[ctx3].items():
                    candidates[word] += cnt * 3
        
        # Bigram (вес 2)
        if len(context) >= 1:
            ctx2 = context[-1]
            if ctx2 in self.bigram:
                for word, cnt in self.bigram[ctx2].items():
                    candidates[word] += cnt * 2
        
        # Unigram fallback (вес 1)
        if not candidates:
            for word, cnt in self.unigram.items():
                candidates[word] += cnt
        
        # Фильтрация по префиксу
        if prefix:
            prefix_lower = prefix.lower()
            candidates = Counter({
                w: s for w, s in candidates.items()
                if w.lower().startswith(prefix_lower)
            })
            
            # Если n-gram ничего не дал с этим префиксом — берём из словаря
            if not candidates:
                candidates = Counter({
                    w: cnt for w, cnt in self.unigram.items()
                    if w.lower().startswith(prefix_lower)
                })
        
        # Топ-K
        total = sum(candidates.values()) or 1
        return [
            (word, round(cnt / total, 4))
            for word, cnt in candidates.most_common(top_k)
        ]


# ── 3. Keyboard interface ─────────────────────────────────────────────────

class ChagataiKeyboard:
    def __init__(self, csv_path: str):
        sentences = load_corpus(csv_path)
        self.model = NgramModel(sentences, n=3)
    
    def suggest(
        self,
        typed_text: str,
        top_k: int = 5,
    ) -> list[str]:
        """
        Главный метод для клавиатуры.
        
        Args:
            typed_text: Всё что ввёл пользователь, включая текущее неполное слово.
                        Пример: "Men seni sev"  ← "sev" не закончено
        
        Returns:
            Список предложений: ["sevdim", "sevaman", "sevär"]
        """
        tokens = typed_text.strip().split()
        
        if not tokens:
            # Ничего не введено — топ частотных слов
            return [w for w, _ in self.model.unigram.most_common(top_k)]
        
        # Последний токен — это текущий префикс (пользователь ещё пишет)
        # Предыдущие — контекст
        last = tokens[-1]
        context = tokens[:-1]
        
        # Определяем: последний токен — это законченное слово или префикс?
        # Если строка заканчивается на пробел — последний токен завершён
        ends_with_space = typed_text.endswith(" ")
        
        if ends_with_space:
            # Пользователь нажал пробел — предсказываем СЛЕДУЮЩЕЕ слово
            context = tokens
            prefix = ""
        else:
            # Пользователь ещё пишет — используем последний токен как префикс
            prefix = last
            context = tokens[:-1]
        
        results = self.model.predict(context, prefix=prefix, top_k=top_k)
        return [word for word, _ in results]
    
    def suggest_with_scores(self, typed_text: str, top_k: int = 5):
        """То же что suggest(), но возвращает (слово, score)."""
        tokens = typed_text.strip().split()
        ends_with_space = typed_text.endswith(" ")
        
        if not tokens:
            return [(w, 1.0) for w, _ in self.model.unigram.most_common(top_k)]
        
        if ends_with_space:
            context, prefix = tokens, ""
        else:
            context, prefix = tokens[:-1], tokens[-1]
        
        return self.model.predict(context, prefix=prefix, top_k=top_k)


# ── 4. Demo ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    kb = ChagataiKeyboard(CSV_PATH)
    
    print("\n" + "="*55)
    print("ТЕСТ: prefix lookup (без контекста)")
    print("="*55)
    test_prefixes = ["sa", "men", "kel", "bu", "Mā", "şay", 'Māwar']
    for p in test_prefixes:
        suggestions = kb.suggest(p, top_k=4)
        print(f"  '{p}' → {suggestions}")
    
    print("\n" + "="*55)
    print("ТЕСТ: следующее слово (контекст + пробел)")
    print("="*55)
    test_contexts = [
        "Men seni ",
        "pādişāh ",
        "yolları ",
        "kiçik "
    ]
    for ctx in test_contexts:
        suggestions = kb.suggest(ctx, top_k=4)
        print(f"  '{ctx.strip()}' → {suggestions}")
    
    print("\n" + "="*55)
    print("ТЕСТ: контекст + текущий префикс")
    print("="*55)
    test_mixed = [
        "Men se",
        "pādişāh bı",
        "ʿArabistānnıñ yo",
    ]
    for t in test_mixed:
        suggestions = kb.suggest_with_scores(t, top_k=4)
        print(f"  '{t}':")
        for word, score in suggestions:
            bar = "█" * int(score * 25)
            print(f"    {word:<25} {score:.4f}  {bar}")
    
    print("\n" + "="*55)
    print("ИНТЕРАКТИВНЫЙ РЕЖИМ (q для выхода)")
    print("="*55)
    while True:
        user_input = input("\n  Введи текст > ")
        if user_input.lower() == "q":
            break
        results = kb.suggest_with_scores(user_input, top_k=5)
        if results:
            print("  Suggestions:")
            for word, score in results:
                print(f"    [{word}]  {score:.4f}")
        else:
            print("  (нет предложений)")

In [ ]:
pd.read_csv('t.csv')['type'].hist()

In [ ]:
pd.read_csv('t.csv').isna().sum()

In [ ]:
dsd

In [ ]:
"""
Chagatai Keyboard v2 — NLTK KneserNey + Edit Distance
======================================================
Улучшения по сравнению с v1:
  1. KneserNeyInterpolated (nltk) вместо ручного n-gram — лучше обобщает на редкие слова
  2. Edit distance (difflib) — исправление опечаток при prefix lookup
  3. Trigram + Bigram + Unigram интерполяция — автоматически через KneserNey
  4. Score = P(слово | контекст) × частота — комбинированный ранкинг
"""

import re
import pandas as pd
from collections import Counter
from difflib import get_close_matches
from nltk.lm import KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline


# ── 1. Загрузка и токенизация ─────────────────────────────────────────────────

CSV_PATH = "/kaggle/working/t.csv"   # ← замени на свой путь

def load_corpus(csv_path: str) -> list[list[str]]:
    df = pd.read_csv(csv_path)
    all_texts = df["chg_Latn"].dropna().tolist()
    sentences = []
    for text in all_texts:
        tokens = text.strip().split()
        tokens = [t for t in tokens if len(t) >= 1]
        if tokens:
            sentences.append(tokens)
    return sentences


# ── 2. KneserNey языковая модель ──────────────────────────────────────────────

class ChagataiLM:
    def __init__(self, sentences: list[list[str]], n: int = 3):
        self.n = n

        # Обучаем KneserNeyInterpolated через nltk
        # padded_everygram_pipeline добавляет <s>/<s> паддинги автоматически
        train_data, vocab = padded_everygram_pipeline(n, sentences)
        self.lm = KneserNeyInterpolated(n)
        self.lm.fit(train_data, vocab)

        # Отдельный Counter для частот (для prefix lookup и ранкинга)
        self.freq = Counter(w for s in sentences for w in s)
        # Словарь реальных слов (без служебных токенов nltk)
        self.vocab_words = sorted(
            {w for w in self.lm.vocab if not w.startswith("<") and not w.startswith("</")},
        )

        print(f"Corpus: {len(sentences)} sentences")
        print(f"Vocab:  {len(self.vocab_words)} unique words")
        print(f"Model:  KneserNeyInterpolated (n={n})")

    def context_score(self, word: str, context: list[str]) -> float:
        """P(word | context) через KneserNey интерполяцию."""
        # nltk ожидает tuple контекста длиной n-1
        ctx = tuple(context[-(self.n - 1):]) if context else ()
        try:
            return self.lm.score(word, ctx)
        except Exception:
            return 0.0

    def predict(
        self,
        context: list[str],
        prefix: str = "",
        top_k: int = 5,
    ) -> list[tuple[str, float]]:
        """
        Предсказывает следующее слово.

        Args:
            context: Предыдущие слова. Пример: ["Men", "seni"]
            prefix:  Введённые буквы. Пример: "sev"  (или "" если пробел)
            top_k:   Количество кандидатов.

        Returns:
            [(слово, score), ...] по убыванию score.
        """
        # Определяем пул кандидатов
        if prefix:
            candidates = self._prefix_candidates(prefix)
        else:
            candidates = self.vocab_words

        if not candidates:
            return []

        # Скорим каждого кандидата: KneserNey × log(freq+1)
        import math
        scored = []
        for word in candidates:
            lm_score = self.context_score(word, context)
            freq_boost = math.log(self.freq.get(word, 0) + 1)
            combined = lm_score * (1 + 0.3 * freq_boost)
            scored.append((word, combined))

        scored.sort(key=lambda x: -x[1])

        # Нормализуем
        total = sum(s for _, s in scored[:top_k]) or 1.0
        return [(w, round(s / total, 4)) for w, s in scored[:top_k]]

    def _prefix_candidates(self, prefix: str) -> list[str]:
        """
        Возвращает слова начинающиеся с prefix.
        Если ничего нет — ищет похожие через edit distance (опечатки).
        """
        prefix_lower = prefix.lower()

        # Точное совпадение по prefix
        exact = [w for w in self.vocab_words if w.lower().startswith(prefix_lower)]

        if exact:
            return exact

        # Fallback: edit distance через difflib
        # get_close_matches ищет слова похожие на prefix среди vocab
        close = get_close_matches(
            prefix,
            self.vocab_words,
            n=10,
            cutoff=0.6,    # similarity threshold
        )
        return close


# ── 3. Keyboard interface ─────────────────────────────────────────────────────

class ChagataiKeyboard:
    def __init__(self, csv_path: str, n: int = 3):
        sentences = load_corpus(csv_path)
        self.lm = ChagataiLM(sentences, n=n)

    def suggest(self, typed_text: str, top_k: int = 5) -> list[str]:
        """
        Главный метод.
        typed_text — всё что ввёл пользователь включая неполное слово.
        Если заканчивается на пробел — предсказываем следующее слово.
        Иначе — дополняем текущее слово.
        """
        results = self.suggest_with_scores(typed_text, top_k)
        return [w for w, _ in results]

    def suggest_with_scores(
        self, typed_text: str, top_k: int = 5
    ) -> list[tuple[str, float]]:
        tokens = typed_text.strip().split()
        ends_with_space = typed_text.endswith(" ")

        if not tokens:
            # Пусто — топ частотных слов
            return [
                (w, round(c / sum(self.lm.freq.values()), 4))
                for w, c in self.lm.freq.most_common(top_k)
            ]

        if ends_with_space:
            context = tokens
            prefix = ""
        else:
            context = tokens[:-1]
            prefix = tokens[-1]

        return self.lm.predict(context, prefix=prefix, top_k=top_k)


# ── 4. Demo ───────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    kb = ChagataiKeyboard(CSV_PATH)

    print("\n" + "="*55)
    print("ТЕСТ: prefix lookup")
    print("="*55)
    test_prefixes = ["sa", "men", "kel", "bu", "Mā", "şay", "Māwar"]
    for p in test_prefixes:
        suggestions = kb.suggest(p, top_k=4)
        print(f"  '{p}' → {suggestions}")

    print("\n" + "="*55)
    print("ТЕСТ: edit distance (опечатки)")
    print("="*55)
    typos = ["salam",   # нет в корпусе — найдёт похожие
             "padisah", # латинизированная опечатка
             "kichk"]   # пропущена буква
    for t in typos:
        suggestions = kb.suggest(t, top_k=3)
        print(f"  '{t}' → {suggestions}")

    print("\n" + "="*55)
    print("ТЕСТ: следующее слово (KneserNey контекст)")
    print("="*55)
    test_contexts = [
        "Men seni ",
        "pādişāh ",
        "yolları ",
        "kiçik ",
    ]
    for ctx in test_contexts:
        results = kb.suggest_with_scores(ctx, top_k=4)
        print(f"\n  '{ctx.strip()}':")
        for word, score in results:
            bar = "█" * int(score * 20)
            print(f"    {word:<28} {score:.4f}  {bar}")

    print("\n" + "="*55)
    print("ТЕСТ: контекст + префикс")
    print("="*55)
    test_mixed = [
        "Men se",
        "pādişāh bı",
        "ʿArabistānnıñ yo",
    ]
    for t in test_mixed:
        results = kb.suggest_with_scores(t, top_k=4)
        print(f"\n  '{t}':")
        for word, score in results:
            bar = "█" * int(score * 20)
            print(f"    {word:<28} {score:.4f}  {bar}")

    print("\n" + "="*55)
    print("ИНТЕРАКТИВНЫЙ РЕЖИМ (q для выхода)")
    print("="*55)
    while True:
        user_input = input("\n  > ").strip()
        if user_input.lower() == "q":
            break
        results = kb.suggest_with_scores(user_input, top_k=5)
        if results:
            for word, score in results:
                print(f"    [{word}]  {score:.4f}")
        else:
            print("  (нет предложений)")

In [ ]:
"""
Chagatai Keyboard v2 — NLTK KneserNey + Edit Distance
======================================================
Улучшения по сравнению с v1:
  1. KneserNeyInterpolated (nltk) вместо ручного n-gram — лучше обобщает на редкие слова
  2. Edit distance (difflib) — исправление опечаток при prefix lookup
  3. Trigram + Bigram + Unigram интерполяция — автоматически через KneserNey
  4. Score = P(слово | контекст) × частота — комбинированный ранкинг
"""

import re
import pandas as pd
from collections import Counter
from difflib import get_close_matches
from nltk.lm import KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline


# ── 1. Загрузка и токенизация ─────────────────────────────────────────────────

CSV_PATH = "/kaggle/working/t.csv"   # ← замени на свой путь

def load_corpus(csv_path: str) -> list[list[str]]:
    df = pd.read_csv(csv_path)
    all_texts = df["chg_Latn"].dropna().tolist()
    sentences = []
    for text in all_texts:
        tokens = text.strip().split()
        tokens = [t for t in tokens if len(t) >= 1]
        if tokens:
            sentences.append(tokens)
    return sentences


# ── 2. KneserNey языковая модель ──────────────────────────────────────────────

class ChagataiLM:
    def __init__(self, sentences: list[list[str]], n: int = 3):
        self.n = n

        # Обучаем KneserNeyInterpolated через nltk
        # padded_everygram_pipeline добавляет <s>/<s> паддинги автоматически
        train_data, vocab = padded_everygram_pipeline(n, sentences)
        self.lm = KneserNeyInterpolated(n)
        self.lm.fit(train_data, vocab)

        # Отдельный Counter для частот (для prefix lookup и ранкинга)
        self.freq = Counter(w for s in sentences for w in s)
        # Словарь реальных слов (без служебных токенов nltk)
        self.vocab_words = sorted(
            {w for w in self.lm.vocab if not w.startswith("<") and not w.startswith("</")},
        )

        print(f"Corpus: {len(sentences)} sentences")
        print(f"Vocab:  {len(self.vocab_words)} unique words")
        print(f"Model:  KneserNeyInterpolated (n={n})")

    def context_score(self, word: str, context: list[str]) -> float:
        """P(word | context) через KneserNey интерполяцию."""
        # nltk ожидает tuple контекста длиной n-1
        ctx = tuple(context[-(self.n - 1):]) if context else ()
        try:
            return self.lm.score(word, ctx)
        except Exception:
            return 0.0

    def predict(
        self,
        context: list[str],
        prefix: str = "",
        top_k: int = 5,
    ) -> list[tuple[str, float]]:
        """
        Предсказывает следующее слово.

        Args:
            context: Предыдущие слова. Пример: ["Men", "seni"]
            prefix:  Введённые буквы. Пример: "sev"  (или "" если пробел)
            top_k:   Количество кандидатов.

        Returns:
            [(слово, score), ...] по убыванию score.
        """
        import math
        from collections import Counter as _Counter

        candidates = self._prefix_candidates(prefix) if prefix else self.vocab_words
        if not candidates:
            return []

        scored = _Counter()

        # Trigram (вес 3) — самый точный если контекст есть в корпусе
        if len(context) >= 2:
            ctx3 = tuple(context[-2:])
            for word in candidates:
                try:
                    scored[word] += self.lm.score(word, ctx3) * 3
                except Exception:
                    pass

        # Bigram (вес 2)
        if len(context) >= 1:
            ctx2 = (context[-1],)
            for word in candidates:
                try:
                    scored[word] += self.lm.score(word, ctx2) * 2
                except Exception:
                    pass

        # Unigram (вес 1) — всегда как базовый fallback
        for word in candidates:
            try:
                scored[word] += self.lm.score(word, ()) * 1
            except Exception:
                pass

        # Frequency boost: частые слова получают небольшой буст
        for word in candidates:
            scored[word] *= (1 + 0.3 * math.log(self.freq.get(word, 0) + 1))

        top = scored.most_common(top_k)
        total = sum(s for _, s in top) or 1.0
        return [(w, round(s / total, 4)) for w, s in top]

    def _prefix_candidates(self, prefix: str) -> list[str]:
        """
        Возвращает слова начинающиеся с prefix.
        Если ничего нет — ищет похожие через edit distance (опечатки).
        """
        prefix_lower = prefix.lower()

        # Точное совпадение по prefix
        exact = [w for w in self.vocab_words if w.lower().startswith(prefix_lower)]

        if exact:
            return exact

        # Fallback: edit distance через difflib
        # get_close_matches ищет слова похожие на prefix среди vocab
        close = get_close_matches(
            prefix,
            self.vocab_words,
            n=10,
            cutoff=0.4,    # 0.4 чтобы ловить слова без диакритики
        )
        return close


# ── 3. Keyboard interface ─────────────────────────────────────────────────────

class ChagataiKeyboard:
    def __init__(self, csv_path: str, n: int = 3):
        sentences = load_corpus(csv_path)
        self.lm = ChagataiLM(sentences, n=n)

    def suggest(self, typed_text: str, top_k: int = 5) -> list[str]:
        """
        Главный метод.
        typed_text — всё что ввёл пользователь включая неполное слово.
        Если заканчивается на пробел — предсказываем следующее слово.
        Иначе — дополняем текущее слово.
        """
        results = self.suggest_with_scores(typed_text, top_k)
        return [w for w, _ in results]

    def suggest_with_scores(
        self, typed_text: str, top_k: int = 5
    ) -> list[tuple[str, float]]:
        tokens = typed_text.strip().split()
        ends_with_space = typed_text.endswith(" ")

        if not tokens:
            # Пусто — топ частотных слов
            return [
                (w, round(c / sum(self.lm.freq.values()), 4))
                for w, c in self.lm.freq.most_common(top_k)
            ]

        if ends_with_space:
            context = tokens
            prefix = ""
        else:
            context = tokens[:-1]
            prefix = tokens[-1]

        return self.lm.predict(context, prefix=prefix, top_k=top_k)


# ── 4. Demo ───────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    kb = ChagataiKeyboard(CSV_PATH)

    print("\n" + "="*55)
    print("ТЕСТ: prefix lookup")
    print("="*55)
    test_prefixes = ["sa", "men", "kel", "bu", "Mā", "şay", "Māwar"]
    for p in test_prefixes:
        suggestions = kb.suggest(p, top_k=4)
        print(f"  '{p}' → {suggestions}")

    print("\n" + "="*55)
    print("ТЕСТ: edit distance (опечатки)")
    print("="*55)
    typos = ["salam",   # нет в корпусе — найдёт похожие
             "padisah", # латинизированная опечатка
             "kichk"]   # пропущена буква
    for t in typos:
        suggestions = kb.suggest(t, top_k=3)
        print(f"  '{t}' → {suggestions}")

    print("\n" + "="*55)
    print("ТЕСТ: следующее слово (KneserNey контекст)")
    print("="*55)
    test_contexts = [
        "Men seni ",
        "pādişāh ",
        "yolları ",
        "kiçik ",
    ]
    for ctx in test_contexts:
        results = kb.suggest_with_scores(ctx, top_k=4)
        print(f"\n  '{ctx.strip()}':")
        for word, score in results:
            bar = "█" * int(score * 20)
            print(f"    {word:<28} {score:.4f}  {bar}")

    print("\n" + "="*55)
    print("ТЕСТ: контекст + префикс")
    print("="*55)
    test_mixed = [
        "Men se",
        "pādişāh bı",
        "ʿArabistānnıñ yo",
    ]
    for t in test_mixed:
        results = kb.suggest_with_scores(t, top_k=4)
        print(f"\n  '{t}':")
        for word, score in results:
            bar = "█" * int(score * 20)
            print(f"    {word:<28} {score:.4f}  {bar}")

    print("\n" + "="*55)
    print("ИНТЕРАКТИВНЫЙ РЕЖИМ (q для выхода)")
    print("="*55)
    while True:
        user_input = input("\n  > ").strip()
        if user_input.lower() == "q":
            break
        results = kb.suggest_with_scores(user_input, top_k=5)
        if results:
            for word, score in results:
                print(f"    [{word}]  {score:.4f}")
        else:
            print("  (нет предложений)")

In [ ]:
"""
Chagatai Keyboard — Improved Version
===================================
Добавлено:
  • Interpolation (trigram + bigram + unigram)
  • Smoothing (alpha)
  • Unicode normalization
  • Char-level fallback
"""

import pandas as pd
import unicodedata
from collections import defaultdict, Counter
from functools import lru_cache


# ── 1. Нормализация ─────────────────────────────────────────────

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return text.lower().strip()


def tokenize(text: str):
    text = normalize(text)
    return [t for t in text.split() if t]


# ── 2. Загрузка ─────────────────────────────────────────────────

def load_corpus(csv_path: str):
    df = pd.read_csv(csv_path)
    texts = df["chg_Latn"].dropna().tolist()
    
    sentences = []
    for t in texts:
        tokens = tokenize(t)
        if tokens:
            sentences.append(tokens)
    
    return sentences


# ── 3. Модель ───────────────────────────────────────────────────

class NgramModel:
    def __init__(self, sentences):
        self.unigram = Counter()
        self.bigram = defaultdict(Counter)
        self.trigram = defaultdict(Counter)
        self.char_model = defaultdict(Counter)
        
        self._build(sentences)
    
    def _build(self, sentences):
        for tokens in sentences:
            for t in tokens:
                self.unigram[t] += 1
            
            for i in range(len(tokens) - 1):
                self.bigram[tokens[i]][tokens[i+1]] += 1
            
            for i in range(len(tokens) - 2):
                ctx = (tokens[i], tokens[i+1])
                self.trigram[ctx][tokens[i+2]] += 1
        
        # char-level fallback
        for word, cnt in self.unigram.items():
            for i in range(len(word)):
                prefix = word[:i+1]
                self.char_model[prefix][word] += cnt
        
        print(f"Unigrams: {len(self.unigram)}")
    
    def _smooth(self, counter, alpha=0.1):
        total = sum(counter.values()) + alpha * len(counter)
        return {k: (v + alpha) / total for k, v in counter.items()}
    
    @lru_cache(maxsize=10000)
    def _predict_cached(self, context_tuple, prefix):
        context = list(context_tuple)
        return self._predict(context, prefix)
    
    def predict(self, context, prefix="", top_k=5):
        return self._predict_cached(tuple(context), prefix)[:top_k]
    
    def _predict(self, context, prefix):
        L3, L2, L1 = 0.6, 0.3, 0.1
        alpha = 0.1
        
        scores = Counter()
        vocab = self.unigram.keys()
        
        ctx2 = context[-1] if len(context) >= 1 else None
        ctx3 = tuple(context[-2:]) if len(context) >= 2 else None
        
        for word in vocab:
            p = 0.0
            
            if ctx3 and ctx3 in self.trigram:
                tri = self.trigram[ctx3]
                p += L3 * ((tri[word] + alpha) / (sum(tri.values()) + alpha * len(tri)))
            
            if ctx2 and ctx2 in self.bigram:
                bi = self.bigram[ctx2]
                p += L2 * ((bi[word] + alpha) / (sum(bi.values()) + alpha * len(bi)))
            
            uni_total = sum(self.unigram.values())
            p += L1 * ((self.unigram[word] + alpha) / (uni_total + alpha * len(self.unigram)))
            
            scores[word] = p
        
        # prefix filter
        if prefix:
            prefix = normalize(prefix)
            filtered = {w: s for w, s in scores.items() if w.startswith(prefix)}
            
            if not filtered:
                # char fallback
                filtered = self.char_model.get(prefix, {})
            
            scores = Counter(filtered)
        
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# ── 4. Keyboard ─────────────────────────────────────────────────

class ChagataiKeyboard:
    def __init__(self, csv_path):
        sentences = load_corpus(csv_path)
        self.model = NgramModel(sentences)
    
    def suggest(self, text, top_k=5):
        tokens = tokenize(text)
        ends_space = text.endswith(" ")
        
        if not tokens:
            return [w for w, _ in self.model.unigram.most_common(top_k)]
        
        if ends_space:
            context, prefix = tokens, ""
        else:
            context, prefix = tokens[:-1], tokens[-1]
        
        results = self.model.predict(context, prefix, top_k)
        return [w for w, _ in results]
    
    def suggest_with_scores(self, text, top_k=5):
        tokens = tokenize(text)
        ends_space = text.endswith(" ")
        
        if not tokens:
            return [(w, 1.0) for w, _ in self.model.unigram.most_common(top_k)]
        
        if ends_space:
            context, prefix = tokens, ""
        else:
            context, prefix = tokens[:-1], tokens[-1]
        
        return self.model.predict(context, prefix, top_k)


# ── 5. Demo ─────────────────────────────────────────────────────

if __name__ == "__main__":
    CSV_PATH = "/kaggle/working/t.csv"
    
    kb = ChagataiKeyboard(CSV_PATH)
    
    print("\n=== TEST ===")
    tests = [
        "men se",
        "men seni ",
        "padishah ",
        "arabistanni yo",
    ]
    
    for t in tests:
        print(f"\nInput: {t}")
        for w, s in kb.suggest_with_scores(t):
            print(f"  {w:<20} {s:.4f}")

In [ ]:
"""
Chagatai Keyboard — Full Demo Version
====================================
Features:
  • Interpolation (trigram + bigram + unigram)
  • Additive smoothing
  • Unicode normalization
  • Char-level fallback
  • Edit distance (typo correction)
  • CLI demo
"""

import pandas as pd
import unicodedata
from collections import defaultdict, Counter
from functools import lru_cache


# ── 1. TEXT PROCESSING ──────────────────────────────────────────

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return text.lower().strip()


def tokenize(text: str):
    text = normalize(text)
    return [t for t in text.split() if t]


# ── 2. LOAD DATA ────────────────────────────────────────────────

def load_corpus(csv_path: str):
    df = pd.read_csv(csv_path)
    texts = df["chg_Latn"].dropna().tolist()
    
    sentences = []
    for t in texts:
        tokens = tokenize(t)
        if tokens:
            sentences.append(tokens)
    
    print(f"Corpus: {len(sentences)} sentences")
    return sentences


# ── 3. EDIT DISTANCE ────────────────────────────────────────────

def edit_distance(a: str, b: str) -> int:
    dp = [[i + j if i * j == 0 else 0 for j in range(len(b) + 1)] for i in range(len(a) + 1)]
    
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )
    return dp[-1][-1]


# ── 4. MODEL ────────────────────────────────────────────────────

class NgramModel:
    def __init__(self, sentences):
        self.unigram = Counter()
        self.bigram = defaultdict(Counter)
        self.trigram = defaultdict(Counter)
        self.char_model = defaultdict(Counter)
        
        self._build(sentences)
    
    def _build(self, sentences):
        for tokens in sentences:
            for t in tokens:
                self.unigram[t] += 1
            
            for i in range(len(tokens) - 1):
                self.bigram[tokens[i]][tokens[i+1]] += 1
            
            for i in range(len(tokens) - 2):
                ctx = (tokens[i], tokens[i+1])
                self.trigram[ctx][tokens[i+2]] += 1
        
        # char-level fallback
        for word, cnt in self.unigram.items():
            for i in range(len(word)):
                prefix = word[:i+1]
                self.char_model[prefix][word] += cnt
        
        print(f"Vocab: {len(self.unigram)} unique words")
    
    @lru_cache(maxsize=10000)
    def _predict_cached(self, context_tuple, prefix):
        return self._predict(list(context_tuple), prefix)
    
    def predict(self, context, prefix="", top_k=5):
        return self._predict_cached(tuple(context), normalize(prefix))[:top_k]
    
    def _predict(self, context, prefix):
        L3, L2, L1 = 0.6, 0.3, 0.1
        alpha = 0.1
        
        scores = Counter()
        vocab = self.unigram.keys()
        
        ctx2 = context[-1] if len(context) >= 1 else None
        ctx3 = tuple(context[-2:]) if len(context) >= 2 else None
        
        uni_total = sum(self.unigram.values())
        
        for word in vocab:
            p = 0.0
            
            # trigram
            if ctx3 and ctx3 in self.trigram:
                tri = self.trigram[ctx3]
                p += L3 * ((tri[word] + alpha) / (sum(tri.values()) + alpha * len(tri)))
            
            # bigram
            if ctx2 and ctx2 in self.bigram:
                bi = self.bigram[ctx2]
                p += L2 * ((bi[word] + alpha) / (sum(bi.values()) + alpha * len(bi)))
            
            # unigram
            p += L1 * ((self.unigram[word] + alpha) / (uni_total + alpha * len(self.unigram)))
            
            scores[word] = p
        
        # prefix filter
        if prefix:
            filtered = {w: s for w, s in scores.items() if w.startswith(prefix)}
            
            if not filtered:
                filtered = self.char_model.get(prefix, {})
            
            scores = Counter(filtered)
        
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    # ── typo correction ───────────────────
    def suggest_corrections(self, word, top_k=3):
        word = normalize(word)
        
        candidates = []
        for w in self.unigram:
            dist = edit_distance(word, w)
            if dist <= 2:
                candidates.append((w, dist, self.unigram[w]))
        
        candidates.sort(key=lambda x: (x[1], -x[2]))
        return [w for w, _, _ in candidates[:top_k]]


# ── 5. KEYBOARD ─────────────────────────────────────────────────

class ChagataiKeyboard:
    def __init__(self, csv_path):
        sentences = load_corpus(csv_path)
        self.model = NgramModel(sentences)
    
    def suggest(self, text, top_k=5):
        tokens = tokenize(text)
        ends_space = text.endswith(" ")
        
        if not tokens:
            return [w for w, _ in self.model.unigram.most_common(top_k)]
        
        if ends_space:
            context, prefix = tokens, ""
        else:
            context, prefix = tokens[:-1], tokens[-1]
        
        return [w for w, _ in self.model.predict(context, prefix, top_k)]
    
    def suggest_with_scores(self, text, top_k=5):
        tokens = tokenize(text)
        ends_space = text.endswith(" ")
        
        if not tokens:
            return [(w, 1.0) for w, _ in self.model.unigram.most_common(top_k)]
        
        if ends_space:
            context, prefix = tokens, ""
        else:
            context, prefix = tokens[:-1], tokens[-1]
        
        return self.model.predict(context, prefix, top_k)


# ── 6. DEMO ─────────────────────────────────────────────────────

if __name__ == "__main__":
    CSV_PATH = "/kaggle/working/t.csv"
    
    kb = ChagataiKeyboard(CSV_PATH)
    
    print("\nModel: Interpolated N-gram (n=3)")
    
    # PREFIX
    print("\n" + "="*55)
    print("ТЕСТ: prefix lookup")
    print("="*55)
    
    for p in ["sa", "men", "kel", "bu", "Mā", "şay", "Māwar"]:
        print(f"  '{p}' → {kb.suggest(p, 4)}")
    
    # EDIT DISTANCE
    print("\n" + "="*55)
    print("ТЕСТ: edit distance (опечатки)")
    print("="*55)
    
    for t in ["salam", "padisah", "kichk"]:
        print(f"  '{t}' → {kb.model.suggest_corrections(t)}")
    
    # NEXT WORD
    print("\n" + "="*55)
    print("ТЕСТ: следующее слово")
    print("="*55)
    
    for ctx in ["Men seni", "pādişāh", "yolları", "kiçik"]:
        print(f"\n  '{ctx}':")
        for word, score in kb.suggest_with_scores(ctx + " ", 4):
            bar = "█" * int(score * 20)
            print(f"    {word:<25} {score:.4f}  {bar}")
    
    # CONTEXT + PREFIX
    print("\n" + "="*55)
    print("ТЕСТ: контекст + префикс")
    print("="*55)
    
    for text in ["Men se", "pādişāh bı", "ʿArabistānnıñ yo"]:
        print(f"\n  '{text}':")
        for word, score in kb.suggest_with_scores(text, 4):
            bar = "█" * int(score * 20)
            print(f"    {word:<25} {score:.4f}  {bar}")
    
    # INTERACTIVE
    print("\n" + "="*55)
    print("ИНТЕРАКТИВНЫЙ РЕЖИМ (q для выхода)")
    print("="*55)
    
    while True:
        text = input("\n  Введи текст > ")
        if text.lower() == "q":
            break
        
        results = kb.suggest_with_scores(text, 5)
        
        if results:
            print("  Suggestions:")
            for w, s in results:
                print(f"    [{w}] {s:.4f}")
        else:
            print("  (нет предложений)")